# Lesson 4: Flood Fill and Connected Components

Once we have a binary image (e.g., from thresholding), a natural next question is: *how many separate blobs are there, and where are they?* Two tools answer this:

- **Flood fill** grows a region from a single seed pixel, spreading to all connected neighbors that share a similar value.
- **Connected component labeling** finds *all* such regions in a binary image at once, labeling each with a unique ID.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## Build a test image with several blobs

We draw a few disconnected shapes plus one pair of shapes that touch, so we can see how connectivity is decided.

In [ ]:
img = np.zeros((150, 200), dtype=np.uint8)
cv2.circle(img, (40, 40), 25, 255, -1)          # blob 1
cv2.rectangle(img, (100, 20), (140, 60), 255, -1)  # blob 2
cv2.circle(img, (60, 110), 20, 255, -1)          # blob 3a
cv2.circle(img, (95, 110), 20, 255, -1)          # blob 3b, touches 3a
cv2.circle(img, (170, 120), 12, 255, -1)         # blob 4, small

plt.imshow(img, cmap='gray')
plt.title('Binary test image')
plt.axis('off')
plt.show()

## Flood fill from a seed point

`cv2.floodFill` starts at a seed pixel and spreads outward to all connected pixels within a tolerance of the seed value, painting them a new color. Here we seed the algorithm with a pixel inside the touching pair of circles: flood fill treats them as *one* region because they are physically connected, even though we drew them as two separate `cv2.circle` calls.

In [ ]:
# floodFill needs a mask 2 pixels larger than the image, and modifies the image in place
im_filled = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
im_mask = np.zeros(np.add(img.shape, (2, 2)), dtype=np.uint8)
seed = (60, 110)  # (x, y) inside the touching pair

cv2.floodFill(im_filled, im_mask, seed, (255, 140, 0))

plt.imshow(im_filled)
plt.scatter(*seed, c='red', s=30, marker='x')
plt.title('Flood fill from one seed (red x)')
plt.axis('off')
plt.show()

## Connected components: label every blob at once

Instead of picking seeds by hand, the connected-component labeling algorithm (`cv2.connectedComponentsWithStats`) scans the whole image and assigns every blob its own integer label. Note that the touching pair of circles is reported as a *single* blob with one label, just as flood fill found it to be a single connected region.

In [ ]:
num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(img, connectivity=8)

# Give each label a distinct random color for visualization
colors = np.array([(0,0,0), (255,0,0), (0,200,0), (0,0,255), (180,180,0)])

im_colored = colors[labels].astype(np.uint8)

plt.imshow(im_colored)
for label in range(1, num_labels):
    cx, cy = centroids[label]
    plt.text(cx, cy, str(label), color='white', ha='center', va='center', fontsize=12, fontweight='bold')
plt.title('Connected components, colored and labeled')
plt.axis('off')
plt.show()

The algorithm also returns handy stats (e.g., bounding box, area, centroid) for free.

In [ ]:
print(f'Found {num_labels - 1} blobs (plus the background as label 0)\n')
print(f'{"label":>5} {"area":>6} {"centroid":>16}')
for label in range(1, num_labels):
    area = stats[label, cv2.CC_STAT_AREA]
    cx, cy = centroids[label]
    print(f'{label:>5} {area:>6} ({cx:6.1f}, {cy:6.1f})')

## Filtering blobs by size

A common use of connected components is to discard small, noise-like blobs and keep only significant ones. Here we choose a threshold that removes the small circle (blob 4).

In [ ]:
min_area = 500
img_minsize = np.zeros_like(img)
for label in range(1, num_labels):
    if stats[label, cv2.CC_STAT_AREA] >= min_area:
        img_minsize[labels == label] = 255

fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
axes[0].imshow(img, cmap='gray')
axes[0].set_title('All blobs')
axes[0].axis('off')
axes[1].imshow(img_minsize, cmap='gray')
axes[1].set_title(f'Blobs with area >= {min_area}')
axes[1].axis('off')
plt.tight_layout()
plt.show()

### Exercise

1. Change `connectivity=8` to `connectivity=4` in `connectedComponentsWithStats`. Construct a binary image (e.g., a diagonal staircase of single pixels) where 4-connectivity and 8-connectivity give a *different* number of components.
2. Use `cv2.floodFill` with a nonzero `loDiff`/`upDiff` tolerance on a grayscale (not binary) image, and describe what changes.